In [1]:
import csv
import pandas as pd
import os
import numpy as np
from datetime import datetime

In [2]:
base_dir = os.getcwd()
oecd_dir = os.path.join(base_dir, "OECD")
update_dir = os.path.join(oecd_dir, "Update_2024")

In [3]:
data_dir = os.path.join(update_dir, "Data")
reshape_dir = os.path.join(update_dir,"reshape")
metadata_dir = os.path.join(update_dir,"metadata")
output_dir = os.path.join(update_dir,"output")
int_output_dir = os.path.join(update_dir,"intermediate_output")

merge_dir = os.path.join(update_dir,"merge")
gcttotal_dir = os.path.join(update_dir,"gct_total")
gct_dir = os.path.join(update_dir,"gct")

act_dir = os.path.join(update_dir,"act_total")
com_dir = os.path.join(update_dir,"com_total")
otp_dir = os.path.join(update_dir,"otp_total")

In [4]:
def get_country_code(filename):
    return filename.replace("pse-","").replace("-2024.xls","")[-3::].upper()

In [5]:
outtotal = pd.read_csv(os.path.join(update_dir, "./mapping/outtotal.csv")).set_index("rawcolumns")["rename"].to_dict()

In [6]:
files = os.listdir(data_dir)

for file in files:
    
    cols_drop = [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,25,
                 26,27,28,29,31,32,33,34,35,36,37,38,39,40,41,42,43,44]

    filepath = os.path.join(data_dir,file)

    country_code = get_country_code(file)
    sheet_names = pd.ExcelFile(filepath).sheet_names
    
    out_file = country_code+'_TOTAL'+ ".csv"
    out_data= pd.DataFrame()
    total_sheet_names = [i for i in sheet_names if 'TOTAL' in i]

    for sheet_name in total_sheet_names:
        cc = sheet_name.split()[0]
        data = pd.read_excel(filepath,sheet_name)

        if True:
#             data = data.set_index(data.columns[0],drop=True)
            data_rs = data[pd.notnull(data[data.columns[0]])]     
            
            data_rs.drop(data_rs.columns[cols_drop],axis=1,inplace=True)
            
            data_rs.rename(columns={'Back to index':'policy_code', 'Unnamed: 21':'commodity_code','Unnamed: 22':'commodity_label',
                                    'Unnamed: 23':'GC_code','Unnamed: 24':'AC_code','Unnamed: 30':'unit'}, inplace=True)
            
            data_rs = data_rs.rename(columns={col: col.split('.')[0] for col in data_rs.columns})
            
            data_rs.rename(columns={'TABLE 1':'policy_label'}, inplace=True)
            
            data_rs['policy_label'] = data_rs['policy_label'].str.lstrip()
            
            data_rs['commodity_code'] = np.where(data_rs['commodity_code']=='OT','AC', data_rs['commodity_code'])
            
            data_rs['commodity_code'] = data_rs['commodity_code'].fillna('') + data_rs['GC_code'].fillna('') + data_rs['AC_code'].fillna('')
            
            data_rs.drop(['GC_code', 'AC_code'], axis=1, inplace=True)                  
                
            data_rs = data_rs[data_rs.unit.notnull()]    
            
#             data_rs= data_rs[~data_rs['policy_code'].str.contains("VP")]
            data_rs= data_rs[~data_rs['policy_code'].str.contains("VP1P")]
            data_rs= data_rs[~data_rs['policy_code'].str.contains("VC")]
            data_rs= data_rs[~data_rs['policy_code'].str.contains("VC1")]
#             data_rs= data_rs[~data_rs['policy_code'].str.contains("PSE")]           
            
                        
            data_rs['country_code'] = country_code   
            
            if 'GBR' in country_code: 
                data_rs.rename(columns={'Unnamed: 45':2017, 'Unnamed: 46':2018,'Unnamed: 47':2019, 'Unnamed: 48':2020, 
                               'Unnamed: 49':2021, 'Unnamed: 50':2022,'Unnamed: 51':2023}, inplace=True)
            else:
                data_rs.rename(columns=outtotal, inplace=True)  
                      
            out_data = pd.concat([out_data,data_rs],ignore_index=True)          
       
            
            out_path = os.path.join(int_output_dir,out_file)
            out_data.to_csv(out_path, index=False)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_rs.drop(data_rs.columns[cols_drop],axis=1,inplace=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_rs.rename(columns={'Back to index':'policy_code', 'Unnamed: 21':'commodity_code','Unnamed: 22':'commodity_label',
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.py

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_rs.drop(data_rs.columns[cols_drop],axis=1,inplace=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_rs.rename(columns={'Back to index':'policy_code', 'Unnamed: 21':'commodity_code','Unnamed: 22':'commodity_label',
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.py

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_rs.rename(columns={'Back to index':'policy_code', 'Unnamed: 21':'commodity_code','Unnamed: 22':'commodity_label',
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_rs.drop(data_rs.columns[cols_drop],axis=1,inplace=True)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1832775669.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.py

In [7]:
products = pd.read_csv(os.path.join(update_dir, "./mapping/commodity_map.csv")).set_index("commodity_code")["commodity_label"].to_dict()

In [8]:
currency = pd.read_csv(os.path.join(update_dir, "./mapping/currency_map.csv")).set_index("ISO3")["CURRENCY"].to_dict()
country_map = pd.read_csv(os.path.join(update_dir, "./mapping/country_map.csv")).set_index("ISO3")["CountryLabel"].to_dict()

In [9]:
files = os.listdir(int_output_dir)

total_df = pd.DataFrame()
for file in files:
    data = pd.read_csv(os.path.join(int_output_dir,file),encoding="latin1")
    total_df = pd.concat([total_df,data],ignore_index=True)
    
total_df.drop(['unit', 'policy_label'], axis=1, inplace=True)

total_df = total_df[total_df.commodity_code.notnull()]

total_df.loc[(total_df['country_code'] == 'CHN') & (total_df['commodity_code'] == 'IF'), 'commodity_code'] = 'IFCHN'
total_df.loc[(total_df['country_code'] == 'CHN') & (total_df['commodity_code'] == 'XF'), 'commodity_code'] = 'XFCHN'
total_df.loc[(total_df['country_code']=='ISR') & (total_df['commodity_code']=='EP'), 'commodity_code'] = 'MN'
total_df.loc[(total_df['country_code'].isin(['IND','CHN','ISR','ZAF'])) & (total_df['commodity_code']=='GN'), 
                  'commodity_code']='PN'

total_df= total_df[~total_df['policy_code'].str.contains("MPS")]
total_df= total_df[~total_df['policy_code'].str.contains("TCT")]
total_df= total_df[~total_df['policy_code'].str.contains("GSSE")]

total_df = pd.melt(total_df,id_vars=['country_code','commodity_code','commodity_label', 
                                     'policy_code'],var_name="year",value_name="value")

total_df = total_df[total_df.value!=0]
print(total_df.shape)

gct_df = total_df[total_df.commodity_label.notnull()]
# gct_df['commodity_type'] = np.nan
print(gct_df.shape)


other_df = total_df[total_df.commodity_label.isnull()]
other_df['commodity_label'] = other_df.commodity_code.map(products)
print(other_df.shape)

total_df = other_df.append(gct_df)

total_df['category'] = np.where(total_df.policy_code.str.contains('MPS'), 'A1', 
                                np.where(total_df.policy_code.str.contains('PO'), 'A2', 
                                        np.where(total_df.policy_code.str.contains('PIV'),'B1', 
                                        np.where(total_df.policy_code.str.contains('PIF'),'B2',
                                        np.where(total_df.policy_code.str.contains('PIS'),'B3',
                                        np.where(total_df.policy_code.str.contains('PC'),'C',
                                        np.where(total_df.policy_code.str.contains('PHR'),'D',
                                        np.where(total_df.policy_code.str.contains('PHNR'),'E',
                                        np.where(total_df.policy_code.str.contains('PN'),'F',
                                        np.where(total_df.policy_code.str.contains('PM'),'G', np.nan))))))))))
total_df.drop(['policy_code'], axis=1, inplace=True)

total_df.year = total_df.year.astype(int)

total_df = total_df[total_df.year>=2005]

total_df = total_df.groupby(['country_code','commodity_code','commodity_label',
                             'category','year']).sum()[['value']].reset_index()

total_df['country_label'] = total_df.country_code.map(country_map)

total_df.rename(columns={'country_label':'Country_Label','country_code':'Country_Code', 
                         'commodity_label':'Commodity_Label', 'commodity_code':'Commodity_Code', 
                     'category':'Category','year':'Year', 'value':"Value"}, inplace=True)

total_df = total_df[['Country_Label','Country_Code', 'Commodity_Label', 'Commodity_Code', 
                     'Category','Year','Value']]

total_df = total_df.sort_values(['Country_Label','Commodity_Label','Category','Year'])

total_df['Value'] = np.where(total_df['Country_Code'].isin(['JPN','KOR']), total_df['Value']*10e8, total_df['Value']*10e5)
print(total_df.shape)

total_df.to_csv(os.path.join(output_dir, 'OECD_Full_Data.csv'), index=False)

(42105, 6)
(10481, 6)
(31624, 6)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\3896253320.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  other_df['commodity_label'] = other_df.commodity_code.map(products)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\3896253320.py:37: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  total_df = other_df.append(gct_df)


(10605, 7)


In [10]:
payment_df = total_df[total_df.Category!='A1']
print(payment_df.shape)

payment_b1b2b3 = payment_df[payment_df.Category.isin(['B1','B2','B3'])]
print(payment_b1b2b3.shape)
print(payment_b1b2b3.Category.unique())
payment_b1b2b3['Category'] ='B'
payment_b1b2b3 = payment_b1b2b3.groupby(['Country_Label','Country_Code','Commodity_Label',
                                         'Commodity_Code','Category','Year']).sum()[['Value']].reset_index()
print(payment_b1b2b3.shape)

payment_oth = payment_df[~(payment_df.Category.isin(['B1','B2','B3']))]
print(payment_oth.shape)

payment_df = payment_b1b2b3.append(payment_oth)
print(payment_df.shape)

payment_df = payment_df[payment_df.Value!=0]
print(payment_df.shape)

payment_df = payment_df.pivot_table(index=['Country_Label','Country_Code','Commodity_Label',
                                               'Commodity_Code','Year'], columns='Category', values=['Value'])
payment_df = payment_df.sort_index(axis=1, level=1)

payment_df.columns = [f'{y}' for x,y in payment_df.columns]
payment_df = payment_df.reset_index()

gct_ac = payment_df[payment_df.Commodity_Code.str.contains('GCT|AC')]
gct_ac['Commodity_Type'] = np.nan

all_else = payment_df[~(payment_df.Commodity_Code.str.contains('GCT|AC'))]
all_else['Commodity_Type'] = np.where(all_else.Commodity_Code.isin(['XE', 'NONMPS']), 'No','Yes')

payment_df = all_else.append(gct_ac)

payment_df = payment_df[['Country_Label', 'Country_Code', 'Commodity_Label', 'Commodity_Code','Commodity_Type',
       'Year', 'A2', 'B', 'C', 'D', 'E', 'F', 'G']]

payment_df.head()
# payment_df.to_csv(os.path.join(output_dir, 'OECD_Payment_Data_Short.csv'), index=False)

(10605, 7)
(5193, 7)
['B1' 'B2' 'B3']
(3177, 7)
(5412, 7)
(8589, 7)
(8001, 7)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\804723791.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  payment_b1b2b3['Category'] ='B'
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\804723791.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  payment_df = payment_b1b2b3.append(payment_oth)
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\804723791.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gc

,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,A2,B,C,D,E,F,G
29,ARGENTINA,ARG,Beef and veal,BF,Yes,2006,NaN,2.163215e+06,NaN,NaN,NaN,NaN,NaN
30,ARGENTINA,ARG,Beef and veal,BF,Yes,2007,NaN,7.790458e+06,NaN,NaN,NaN,NaN,NaN
31,ARGENTINA,ARG,Beef and veal,BF,Yes,2008,NaN,1.193994e+07,NaN,NaN,NaN,NaN,NaN
32,ARGENTINA,ARG,Beef and veal,BF,Yes,2009,NaN,2.047745e+07,NaN,NaN,NaN,NaN,NaN
33,ARGENTINA,ARG,Beef and veal,BF,Yes,2010,NaN,1.850057e+07,NaN,NaN,NaN,NaN,NaN


In [11]:
gct_ac.Commodity_Label.unique()

array(['All crops', 'All fruits and vegetables', 'Beef and milk',
       'Olive  sweet citrus  nuts  legumes  dair', 'Unallocated',
       'All livestock', 'Horticultural crops', 'Other crops', 'PK & EG',
       'Ruminants', 'All arable crops', 'all -SM', 'poultry and eggs',
       'vegetables', 'Grains', 'WT RI MA SB CT RP', 'COP',
       'Milk and beef', 'Protein crops', 'DY  BF', 'SH  WL', 'Oilseeds',
       'Pulses', 'RI  MA  SB', 'Beef and sheep', 'Fruit excluding citrus',
       'Oranges and grapefruit', 'Vegetables',
       'wheat  barley and soybeans', 'Feed crops',
       'Greenhouse vegetables', 'Beef and Milk', 'Beef and Pigmeat',
       'Bovine  ovine  caprine  porcine and hone',
       'Maize  beans and rice', 'maize and beans', 'Feed', 'Tubers',
       'Soybeans and rapeseed', 'Beef and veal  sheep',
       'All crops except wine', 'Grains&Oilseeds', 'Leguminous crops',
       'hazelnuts  tobacco', 'wheat  sugar  cotton  sunflower',
       'All except milk and meat', 'Gra

In [12]:
oecd_full = total_df.pivot_table(index=['Country_Label','Country_Code','Commodity_Label',
                                               'Commodity_Code','Year'], columns='Category', values=['Value'])
oecd_full = oecd_full.sort_index(axis=1, level=1)

oecd_full.columns = [f'{y}' for x,y in oecd_full.columns]
oecd_full = oecd_full.reset_index()

gct_ac = oecd_full[oecd_full.Commodity_Code.str.contains('GCT|AC')]
gct_ac['Commodity_Type'] = np.nan

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\28579894.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gct_ac['Commodity_Type'] = np.nan


In [13]:
gct_ac = gct_ac.replace({'Commodity_Label':{'all -SM':'All supply managed commodities',
       'poultry and eggs':'Poultry and eggs', 
       'wheat  barley and soybeans':'Wheat, barley and soybeans', 
       'maize and beans':'Maize and beans', 
        'Grains&Oilseeds':'Grains and oilseeds', 
       'hazelnuts  tobacco':'Hazelnuts and tobacco', 
        'wheat  sugar  cotton  sunflower':'Wheat, sugar, cotton and sunflower', 
        'PK & EG': 'Pig meat and eggs',
        'WT RI MA SB CT RP':'Wheat, rice, maize, soybeans, cotton and rapeseed', 
        'SH  WL':'Sheep meat and wool',
        'Milk and beef':'Beef and milk', 
        'Beef and Milk':'Beef and milk', 
        'Beef and sheep': 'Beef and sheep meat',
        'Sheepmeat  wool  beef and milk':'Sheep meat, wool, beef and milk', 
        'sorghum  maize  oilseeds':'Sorghum, maize and oilseeds',
        'Beef and Pigmeat':'Beef and pig meat', 
        'All fruits and vegetables':'Fruits and vegetables', 
        'Vegetables':'Vegetables excluding roots and tubers',
        'vegetables':'Vegetables excluding roots and tubers',
        'COP':'Cereals, oilseeds and protein crops', 
        'DY  BF':'Beef and milk', 
        'Feed':'Feed crops', 'Tree and Vineyard':'Tree and vineyard', 
        'Fruit excluding citrus':'All Fruits -(Oranges & Grapefruit)', 
        'Beef and veal  sheep': 'Beef and sheep meat', 
        'Maize  beans and rice':'Maize, beans and rice',
        'Bovine  ovine  caprine  porcine and hone':'Bovine, ovine, caprine, porcine and honey',
        'RI  MA  SB':'Rice, maize and soybeans',
        'Horticultural crops':'Horticulture',
        'Fodder crops':'Feed crops',
        'Olive  sweet citrus  nuts  legumes  dair': 'Olive, citrus fruits, nuts, legumes, cheese and garlic'
        }})

In [14]:
all_else = oecd_full[~(oecd_full.Commodity_Code.str.contains('GCT|AC'))]
all_else['Commodity_Type'] = np.where(all_else.Commodity_Code.isin(['XE', 'NONMPS']), 'No','Yes')

oecd_full = all_else.append(gct_ac)

oecd_full.Country_Code = np.where(oecd_full.Country_Code=='EU', 'EUR', oecd_full.Country_Code)

oecd_full.Country_Label = oecd_full.Country_Label.str.lower().str.title()

oecd_full.Country_Label = np.where(oecd_full.Country_Label=='Turkey', "Türkiye", oecd_full.Country_Label)

print(oecd_full.shape)

(5700, 15)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\2022770412.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_else['Commodity_Type'] = np.where(all_else.Commodity_Code.isin(['XE', 'NONMPS']), 'No','Yes')
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\2022770412.py:4: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  oecd_full = all_else.append(gct_ac)


In [15]:
gct_com = ['All crops except wine',
'All crops, cattle and sheep',
'Leguminous crops',
'Protein crops',
'Cereals, oilseeds and protein crops',
'Alternative crops',
'Non-insured crops',
'Feed crops',
'Soybeans, wheat and maize',
'Wheat, rice, maize, soybeans, cotton and rapeseed',
'Pulses',
'Wheat, barley and soybeans',
'Maize and beans',
'Sorghum, maize and oilseeds',
'Wheat, sugar, cotton and sunflower',
'Biomass',
'Fruits',
'All Fruits -(Oranges & Grapefruit)',
'Oranges and grapefruit',
'Citruses',
'Hazelnuts and tobacco',
'Tree and vineyard',
'Greenhouse vegetables',
'Tubers',
'Horticulture',
'Fruits, flowers, industrial crops',
'Grains and oilseeds',
'Beef and milk',
'Beef and sheep meat',
'Sheep meat, wool, beef and milk',
'Sheep meat and wool',
'Pig meat and eggs',
'Poultry and eggs',
'Poultry and pig',
'Beef and pig meat',
'All except milk and meat',
'All supply managed commodities',
'Fruits and vegetables', 
'Vegetables excluding roots and tubers', 
'Rice, maize and soybeans',
'Bovine, ovine, caprine, porcine and honey',
'Maize, beans and rice', 
'Soybeans and rapeseed',
'Horticultural crops',
'Fodder crops', 
'Olive, citrus fruits, nuts, legumes, cheese and garlic'
]

In [16]:
gct_com_map = {'All crops except wine':'GCT11CHE',
'All crops, cattle and sheep':'GCT12CHE',
'Leguminous crops':'GCT13CHE',
'Protein crops':'GCTPRO',
'Cereals, oilseeds and protein crops':'GCT11EUR',
'Alternative crops':'GCT12MEX',
'Non-insured crops':'GCT10USA',
'Feed crops':'GCTFD',
'Soybeans, wheat and maize':'GCT10CHN',
'Wheat, rice, maize, soybeans, cotton and rapeseed':'GCT11CHN',
'Pulses':'GCT10IND',
'Wheat, barley and soybeans':'GCT10JPN',
'Maize and beans':'GCT10MEX',
'Sorghum, maize and oilseeds':'GCT11MEX',
'Wheat, sugar, cotton and sunflower':'GCT11TUR',
'Biomass':'GCT12USA',
'Fruits':'GCT13MEX',
'All Fruits -(Oranges & Grapefruit)':'GCT10ISR',
'Oranges and grapefruit':'GCT11ISR',
'Citruses':'GCT14MEX',
'Hazelnuts and tobacco':'GCT12TUR',
'Tree and vineyard':'GCT11USA',
'Greenhouse vegetables':'GCT12KAZ',
'Tubers':'GCT11NOR',
'Horticulture':'GCTHORT',
'Fruits, flowers, industrial crops':'GCT15MEX',
'Grains and oilseeds':'GCTGNOS',
'Beef and milk':'GCTBFMK',
'Beef and sheep meat':'GCTBFSH',
'Sheep meat, wool, beef and milk':'GCT10NZL',
'Sheep meat and wool':'GCT10ISL',
'Pig meat and eggs':'GCT10AUS',
'Poultry and eggs':'GCT11CAN',
'Poultry and pig':'GCT11RUS',
'Beef and pig meat':'GCT11KOR',
'All except milk and meat':'GCT10UKR',
'All supply managed commodities':'GCT10CAN', 
'Vegetables excluding roots and tubers':'GCTVEG', 
'Fruits and vegetables':'FV',
'Maize, beans and rice':'GCT16MEX',
'Bovine, ovine, caprine, porcine and honey':'GCT17MEX',
'Rice, maize and soybeans':'GCT10IDN', 
'Soybeans and rapeseed':'GCT13RUS', 
'Olive, citrus fruits, nuts, legumes, cheese and garlic': 'GCT11ARG'
}

In [17]:
gct_df = oecd_full[oecd_full['Commodity_Label'].isin(gct_com)]
gct_df['Commodity_Code']=gct_df.Commodity_Label.map(gct_com_map)
gct_df.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\1230701898.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gct_df['Commodity_Code']=gct_df.Commodity_Label.map(gct_com_map)


(603, 15)

In [18]:
oecd_full.Country_Code.unique()

array(['ARG', 'AUS', 'BRA', 'CAN', 'CHN', 'COL', 'CRI', 'EUR', 'ISL',
       'IND', 'IDN', 'ISR', 'JPN', 'KAZ', 'KOR', 'MEX', 'NOR', 'PHL',
       'RUS', 'CHE', 'TUR', 'UKR', 'GBR', 'USA', 'VNM', 'CHL', 'NZL',
       'ZAF'], dtype=object)

In [19]:
# relabeling commodity label and commodity codes 
xe = oecd_full[oecd_full.Commodity_Code=='XE']
xe['Commodity_Label'] = xe.Commodity_Label + ' - ' + xe['Country_Label']
xe['Commodity_Code'] = xe.Commodity_Code + xe['Country_Code']
print(xe.shape)

oth_crop = oecd_full[oecd_full.Commodity_Code=='GCT5']
oth_crop['Commodity_Label'] = oth_crop.Commodity_Label + ' - ' + oth_crop['Country_Label']
oth_crop['Commodity_Code'] = oth_crop.Commodity_Code + oth_crop['Country_Code']
print(oth_crop.shape)

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\4012635000.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  xe['Commodity_Label'] = xe.Commodity_Label + ' - ' + xe['Country_Label']


(316, 15)
(78, 15)


C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\4012635000.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  xe['Commodity_Code'] = xe.Commodity_Code + xe['Country_Code']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\4012635000.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oth_crop['Commodity_Label'] = oth_crop.Commodity_Label + ' - ' + oth_crop['Country_Label']
C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_10404\4012635000.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy 

In [20]:
ocrop = oecd_full[oecd_full.Commodity_Label=='Other crops']
# ocrop.Country_Label.unique()

In [21]:
nongct_df = oecd_full[~(oecd_full['Commodity_Label'].isin(gct_com))]
nongct_df = nongct_df[~(nongct_df.Commodity_Code.isin(['XE', 'GCT5']))]
print(nongct_df.shape)

(4703, 15)


In [22]:
df_append = [nongct_df, xe, oth_crop, gct_df]
oecd_relab = pd.concat(df_append)

print(oecd_full.shape)
print(oecd_relab.shape)

(5700, 15)
(5700, 15)


In [23]:
oecd_relab['B'] = oecd_relab['B1'].fillna(0)+oecd_relab['B2'].fillna(0)+oecd_relab['B3'].fillna(0)

oecd_relab['Country_Code'] = np.where(oecd_relab.Country_Code=='EU','EUR',oecd_relab.Country_Code)

oecd_relab = oecd_relab[['Country_Label', 'Country_Code', 'Commodity_Label', 'Commodity_Code','Commodity_Type',
       'Year', 'A2','B', 'B1','B2','B3', 'C', 'D', 'E', 'F', 'G']]

oecd_relab.to_csv(os.path.join(output_dir, 'OECD_Payment_Data.csv'), index=False, encoding="utf-8-sig")